In [ ]:
import pandas as pd

train = pd.read_csv("TRAIN.csv")
test = pd.read_csv("TEST.csv")

print(train.shape)
print(test.shape)

47 features and one output label

In [ ]:
test.head()

In [ ]:
train.head()

In [ ]:
train.shape

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
train.corr()["Class"].sort_values(ascending=False)

Features F01, F09, F29, F19, and F21 show the strongest positive correlation with the faulty class. This suggests these variables may capture abnormal operational behaviour of the device and could be important predictors for fault detection.

In [ ]:
train.isnull().sum()

In [ ]:
train["Class"].value_counts()

this is moderately balanced dataset.

Normal  ≈ 60%
Faulty  ≈ 40%


In [ ]:
train.hist(figsize=(20,20))

In [ ]:
import seaborn as sns

sns.boxplot(x=train["Class"], y=train["F01"])

In [ ]:
from sklearn.model_selection import train_test_split

X = train.drop("Class", axis=1)
y = train["Class"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

When we split the dataset, we want both training and validation sets to have the same class distribution as the original dataset.So after using stratify the distribution remains same which is :-

0 → 60%

1 → 40%

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [ ]:
X_train.head()

In [ ]:
X_val.head()

In [ ]:
y_train.head()

In [ ]:
y_val.head()

In [ ]:
train.var().sort_values(ascending=False)

Variance analysis reveals that features F31, F38, F37, F32, and F33 exhibit significantly higher variability compared to other sensors. This indicates these parameters capture strong fluctuations in device behavior, which may correspond to operational instability or fault conditions.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_val_scaled)

accuracy = accuracy_score(y_val, y_pred)

print("Validation Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_val, y_pred))

Normal devices correctly predicted as normal.i.e.4908

Normal devices predicted as faulty.i.e.385

False negatives (FN) — faults predicted as normal.
i.e.1749[very dangerous]

Faulty devices correctly predicted.i.e 1714

In [ ]:
from sklearn.metrics import precision_score, recall_score

precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)

print("Precision:", precision)
print("Recall:", recall)

The baseline logistic regression model showed good precision but relatively low recall for the faulty class. To improve fault detection capability, class weighting was introduced to penalize misclassification of faulty devices more heavily.

-------------------------------------------------------------

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_val_scaled)

accuracy = accuracy_score(y_val, y_pred)

print("Validation Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_val, y_pred))


In [ ]:
from sklearn.metrics import precision_score, recall_score

precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)

print("Precision:", precision)
print("Recall:", recall)

The baseline logistic regression model exhibited relatively low recall for the faulty class, indicating that many faults were not detected. By introducing class balancing, the model became more sensitive to the faulty condition, increasing the number of correctly detected faults. Although the overall accuracy slightly decreased, the reduction in false negatives significantly improves the reliability of the fault detection system.

--------------------------------------------------------------

Features pushing prediction toward Fault (Class = 1)

In [ ]:

coef = pd.Series(model.coef_[0], index=X.columns)
coef.sort_values(ascending=False).head(10)

--------------------------------------------
Features pushing prediction toward Normal (Class = 0)

In [ ]:
coef.sort_values().head(10)

The learned coefficients reveal that features F05, F09, and F04 have the strongest positive influence on fault prediction, suggesting that these sensors capture key abnormal behavior patterns. Conversely, features such as F25 and F06 show strong negative coefficients, indicating their association with normal operating conditions.

------------------------------------------------------------

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_val)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

print("Accuracy:", accuracy_score(y_val, y_pred_rf))
print(confusion_matrix(y_val, y_pred_rf))

In [ ]:
from sklearn.metrics import precision_score, recall_score

precision = precision_score(y_val, y_pred_rf)
recall = recall_score(y_val, y_pred_rf)

print("Precision:", precision)
print("Recall:", recall)

--------------------------
To ensure that the model generalizes well and is not overfitting the training data, a 5-fold cross-validation procedure was performed.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf, X, y, cv=5, scoring="accuracy")

print(scores)
print("Mean accuracy:", scores.mean())

The model demonstrates strong generalization ability, as shown by stable performance across all cross-validation folds.

--------------------------------------------------------------

Feature Importance Analysis

In [ ]:


importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(10)

Logistic Regression: captures linear influence of features.

Random Forest: captures non-linear relationships and feature interactions.

This explains why F01 appears important in Random Forest even though its effect in Logistic Regression is just a single linear coefficient.

---------------------------------
Train on Full Training Data

In [ ]:
rf_final = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_final.fit(X, y)

In [ ]:
test = pd.read_csv("TEST.csv")

In [ ]:
test_ids = test["ID"]

In [ ]:
X_test = test.drop(columns=["ID"])
predictions = rf_final.predict(X_test)

In [ ]:
submission = pd.DataFrame({
    "ID": test_ids,
    "CLASS": predictions
})

submission.to_csv("FINAL.csv", index=False)

In [ ]:
from sklearn.metrics import f1_score

f1 = f1_score(y_val, y_pred_rf)

print("F1 Score:", f1)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
y_prob = rf.predict_proba(X_test)[:,1]
import numpy as np

threshold = 0.3
y_pred_new = (y_prob >= threshold).astype(int)

from sklearn.metrics import accuracy_score, precision_score, recall_score

print("Accuracy:", accuracy_score(y_test, y_pred_new))
print("Precision:", precision_score(y_test, y_pred_new))
print("Recall:", recall_score(y_test, y_pred_new))

from sklearn.metrics import f1_score

thresholds = np.arange(0.1, 0.9, 0.01)

best_threshold = 0
best_f1 = 0

for t in thresholds:
    
    y_pred = (y_prob >= t).astype(int)
    score = f1_score(y_test, y_pred)
    
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)

from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

plt.plot(thresholds, precision[:-1], label="Precision")
plt.plot(thresholds, recall[:-1], label="Recall")

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.show()

This helps visualize the trade-off between precision and recall.

------------------------------------------------------------

In [ ]:
thresholds = np.arange(0.0,1.01,0.01)

f1_scores = []

for t in thresholds:
    
    y_pred_temp = (y_prob >= t).astype(int)
    
    f1 = f1_score(y_test, y_pred_temp)
    
    f1_scores.append(f1)

best_threshold = thresholds[np.argmax(f1_scores)]

print("Best Threshold:", best_threshold)
print("Best F1 Score:", max(f1_scores))

In [ ]:
y_pred_final = (y_prob >= 0.4).astype(int)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_final))
print("Precision:", precision_score(y_test, y_pred_final))
print("Recall:", recall_score(y_test, y_pred_final))
print("F1 Score:", f1_score(y_test, y_pred_final))

print(confusion_matrix(y_test, y_pred_final))

In [ ]:
errors = X_test[y_test != y_pred_final]
print(errors.shape)
errors.head()


ROC-AUC score

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label="ROC Curve (AUC = %0.4f)" % roc_auc)
plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

plt.legend()
plt.show()

print("AUC Score:", roc_auc)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train, y_train)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train, y_train)

In [ ]:
test_ids = test["ID"]

X_test_final = test.drop("ID", axis=1)

test_predictions = rf.predict(X_test_final)

In [ ]:

submission = pd.DataFrame({
    "ID": test_ids,
    "CLASS": test_predictions
})

submission.to_csv("FINAL.csv", index=False)